In [7]:
# %pip install numpy
# %pip install pandas
# %pip install pymongo
# %pip install scikit-learn

In [19]:
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

def get_data_from_mongodb(uri = 'mongodb://localhost:27017', db_name = 'iot_project', collection = 'measurements'):
    df = pd.DataFrame()
    try:
        client = MongoClient(uri, serverSelectionTimeoutMS=1000)
        client.admin.command("ping")
        print("Connection successful")

        db = client[db_name]
        collection = db[collection]

        cursor = collection.find({})
        df = pd.DataFrame(list(cursor))
        print(list(df.columns))
        return df
    except ConnectionFailure:
        print("Connection failed")

def clean_data(df, add_moving_avg_cols = True):
    #delete high correlation and irrelevant columns
    df = df.drop(columns=['_id', 'node_id', 'raw_temperature', 'raw_humidity', 'raw_line'])

    #delete rows with more than 90% missing data
    df = df.dropna(thresh=int(0.9 * len(df.columns)), axis=0)

    #delete duplicate rows
    df = df.drop_duplicates()

    #delete statistic outliers
    for column in df.select_dtypes(include=['float64', 'int64']).columns:
        q1 = df[column].quantile(0.25)
        q3 = df[column].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        df = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

    #delete realistic outliners
    df.loc[
        (df["temperature"] < -20) |
        (df["temperature"] > 60),
        "temperature"
    ] = np.nan

    df.loc[
        (df["humidity"] < 0) |
        (df["temperature"] > 100),
        "temperature"
    ] = np.nan

    #add moving avg columns to the df
    # if add_moving_avg_cols:
    #     df["temperature_moving_avg"] = df["temperature"].rolling(window=3).mean()
    #     df["humidity_moving_avg"] = df["humidity"].rolling(window=3).mean()

    #fill in blanks
    df[['temperature', 'humidity']] = df[['temperature', 'humidity']].interpolate()

    return df

In [20]:
raw_data = get_data_from_mongodb()

raw_data

Connection successful
['_id', 'timestamp', 'node_id', 'temperature', 'humidity', 'count', 'raw_temperature', 'raw_humidity', 'raw_line']


,_id,timestamp,node_id,temperature,humidity,count,raw_temperature,raw_humidity,raw_line
0,6a0b41f94d23a6034fbe0721,2026-05-18 16:44:41.444,1,26.64,45.17,294,6624,1332,"node=1,temp=6624,humidity=1332,count=294"
1,6a0b42224d23a6034fbe0722,2026-05-18 16:45:22.478,1,26.62,45.20,296,6622,1333,"node=1,temp=6622,humidity=1333,count=296"
2,6a0b42374d23a6034fbe0723,2026-05-18 16:45:43.017,1,26.60,45.23,297,6620,1334,"node=1,temp=6620,humidity=1334,count=297"
3,6a0b424c6b194568a8d38e22,2026-05-18 16:46:04.975,1,26.58,45.36,298,6618,1338,"node=1,temp=6618,humidity=1338,count=298"
4,6a0b42606b194568a8d38e23,2026-05-18 16:46:24.053,1,26.56,45.46,299,6616,1341,"node=1,temp=6616,humidity=1341,count=299"
...,...,...,...,...,...,...,...,...,...
15321,6a10d84c001f7d2d8940e2ad,2026-05-22 22:27:24.803,1,26.33,46.29,17765,6593,1367,"node=1,temp=6593,humidity=1367,count=17765"
15322,6a10d861001f7d2d8940e2ae,2026-05-22 22:27:45.321,1,26.34,46.19,17766,6594,1364,"node=1,temp=6594,humidity=1364,count=17766"
15323,6a10d875001f7d2d8940e2af,2026-05-22 22:28:05.857,1,26.37,46.23,17767,6597,1365,"node=1,temp=6597,humidity=1365,count=17767"
15324,6a10d88a001f7d2d8940e2b0,2026-05-22 22:28:26.396,1,26.38,46.23,17768,6598,1365,"node=1,temp=6598,humidity=1365,count=17768"


In [21]:
data = clean_data(raw_data)
data

,timestamp,temperature,humidity,count,temperature_moving_avg,humidity_moving_avg
514,2026-05-19 10:00:41.691,25.83,48.16,2958,NaN,NaN
515,2026-05-19 10:01:02.220,25.89,47.94,2959,NaN,NaN
516,2026-05-19 10:01:22.748,26.00,47.69,2960,25.906667,47.930000
517,2026-05-19 10:01:43.276,26.05,47.67,2961,25.980000,47.766667
518,2026-05-19 10:02:03.811,26.07,47.54,2962,26.040000,47.633333
...,...,...,...,...,...,...
14764,2026-05-22 19:16:49.006,26.50,47.40,17208,26.463333,47.446667
14765,2026-05-22 19:17:09.546,26.47,47.33,17209,26.473333,47.393333
14766,2026-05-22 19:17:30.067,26.44,47.32,17210,26.470000,47.350000
14767,2026-05-22 19:17:50.606,26.45,47.22,17211,26.453333,47.290000
